In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import sys
import importlib

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# Notebook location:
# THESIS/z.parcels_postprocessing/notebooks/analyze_particles.ipynb
NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent

for path in [
    PROJECT_DIR,
    PROJECT_DIR / "z.flow_postprocessing",
    PROJECT_DIR / "z.parcels_postprocessing",
]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)


import scripts.fieldset as pfs
import scripts.run as prun
import scripts.metrics_clustering as mclus
import scripts.metrics_flow_coupling as mcouple
import scripts.notebook_helpers as phelp
import theme.plot_theme as ptheme

for module in [pfs, prun, mclus, mcouple, phelp, ptheme]:
    importlib.reload(module)

ptheme.apply_theme()

print(f"Notebook dir : {NB_DIR}")
print(f"Project dir  : {PROJECT_DIR}")
print(f"run.py loaded from: {Path(prun.__file__).resolve()}")
print(f"clustering loaded from: {Path(mclus.__file__).resolve()}")


In [ ]:
# ============================================================
# 2. CASE / COLLECTION SETTINGS
# ============================================================

case_name = "parcels_sea_win_spinupsim_L1"
RUN_CASE = case_name

INPUT_DIR = (NB_DIR / "../data/input").resolve()
DATA_FILE = (INPUT_DIR / f"{case_name}.nc").resolve()
CASE_RESULTS_DIR = (NB_DIR / f"../results/{case_name}").resolve()
METADATA_DIR = CASE_RESULTS_DIR / "metadata"
METRICS_DIR = CASE_RESULTS_DIR / "metrics"
FIG_DIR = CASE_RESULTS_DIR / "figures"

# Set to None to automatically use the newest *_config.json in metadata/.
RUN_COLLECTION_ID = None
# RUN_COLLECTION_ID = "passive_plus_mrsm_k0_release_t0000"

if RUN_COLLECTION_ID is None:
    configs = sorted(METADATA_DIR.glob("*_config.json"), key=lambda p: p.stat().st_mtime)
    if len(configs) == 0:
        raise FileNotFoundError(f"No run collection config found in {METADATA_DIR}")
    CONFIG_JSON = configs[-1]
else:
    CONFIG_JSON = METADATA_DIR / f"{RUN_COLLECTION_ID}_config.json"

if not CONFIG_JSON.exists():
    raise FileNotFoundError(f"Could not find run collection config:\n{CONFIG_JSON}")

print(f"Using collection config: {CONFIG_JSON}")


# ============================================================
# 3. CLUSTERING / RESIDENCE SETTINGS
# ============================================================

SAVE_FIGURES = False
SAVE_METRICS = True

# Compact plot selections.
SNAPSHOT_OBS_SELECTIONS = [0, "last"]
PDF_OBS_SELECTIONS = [0, "last"]

PLOT_TRAJECTORY_SUBSET_PER_CLASS = True
PLOT_POSITION_SNAPSHOTS_COMBINED = True
PLOT_VORONOI_SNAPSHOTS = True
PLOT_VORONOI_PDF = True
PLOT_RESIDENCE_PDF = True

# Voronoï PDF / cluster definition.
# "pdf_intersection" uses the first intersection with the 2D Poisson-Voronoï PDF.
# "fixed" uses FIXED_CLUSTER_THRESHOLD_AREA_NORM.
VORONOI_THRESHOLD_METHOD = "pdf_intersection"
FIXED_CLUSTER_THRESHOLD_AREA_NORM = 0.5
VORONOI_LOG_AREA = True
VORONOI_BINS = np.linspace(-4.0, 4.0, 81)  # bins for ln(A/<A>)

# Residence-time analysis uses all output times.
COMPUTE_RESIDENCE_FOR_ALL_OBS = True
MIN_RESIDENCE_DURATION_OBS = 1

# Easy plot-label override. Keys are particle tags from the metadata table below.
LABEL_OVERRIDES = {
    "passive": "Passive particles",
    "mrsm_B0p68_d0p25m_St0p002371_dragconstant_C1": "St=0.0024 (B=0.68, d=0.25m)",
}


In [ ]:
# ============================================================
# 4. LOAD TRAJECTORIES + DOMAIN
# ============================================================

trajectories_all, collection_config = phelp.load_trajectory_collection_from_config(
    CONFIG_JSON,
    loader=prun.load_trajectories,
    load=False,
)

LEVEL_INDICES = tuple(collection_config["level_indices"])
RELEASE_TIME_INDEX = int(collection_config["release_time_index"])
RUNTIME_DAYS = float(collection_config["runtime_days"])
TIME_STEP_SECONDS = int(collection_config["time_step_seconds"])
PERIODIC = bool(collection_config["periodic"])
level_tag = "k" + "-".join(str(k) for k in LEVEL_INDICES)

_, _, ds_parcels = pfs.build_fieldset(
    DATA_FILE,
    surface_only=True,
    mesh="flat",
    level_indices=LEVEL_INDICES,
    time_step_seconds=TIME_STEP_SECONDS,
    periodic=PERIODIC,
    add_derivatives=False,
)

Lx, Ly = phelp.get_periodic_domain_lengths(ds_parcels)
domain = mclus.domain_from_parcels_dataset(ds_parcels)

print("\nLoaded run collection")
print(f"  case           : {case_name}")
print(f"  collection     : {collection_config['run_collection_id']}")
print(f"  n classes      : {len(trajectories_all)}")
print(f"  runtime days   : {RUNTIME_DAYS}")
print(f"  domain         : {domain.Lx/1000:.1f} x {domain.Ly/1000:.1f} km")


In [ ]:
# ============================================================
# 5. SELECT / LABEL PARTICLE CLASSES
# ============================================================

# Use None to include all. Examples:
# INCLUDE_TAGS = ["passive", "mrsm_B0p68_d0p25m_St0p002371_dragconstant_C1"]
# INCLUDE_PARTICLE_CLASSES = ["passive", "mr_sm"]
# STOKES_RANGE = (0.0, 0.01)
# DIAMETER_RANGE_M = (0.05, 0.30)
# DRAG_CORRECTIONS = ["constant"]

INCLUDE_TAGS = None
INCLUDE_PARTICLE_CLASSES = None
STOKES_RANGE = None
DIAMETER_RANGE_M = None
DRAG_CORRECTIONS = None

trajectories = phelp.filter_trajectories(
    trajectories_all,
    include_tags=INCLUDE_TAGS,
    include_particle_classes=INCLUDE_PARTICLE_CLASSES,
    stokes_range=STOKES_RANGE,
    diameter_range_m=DIAMETER_RANGE_M,
    drag_corrections=DRAG_CORRECTIONS,
)

trajectories = phelp.apply_label_overrides(trajectories, LABEL_OVERRIDES)

if len(trajectories) == 0:
    raise RuntimeError("No particle classes selected.")

metadata_table = phelp.particle_metadata_table(trajectories)
display(metadata_table)

class_style_map = phelp.build_class_style_map(trajectories)


In [ ]:
# ============================================================
# 6. COMPACT TRAJECTORY / POSITION FIGURES
# ============================================================

TRAJ_FIG_DIR = FIG_DIR / "trajectory_snapshots"

if PLOT_TRAJECTORY_SUBSET_PER_CLASS:
    trajectory_subset_figures = phelp.plot_trajectory_subset_per_class(
        trajectories,
        Lx=Lx,
        Ly=Ly,
        run_title_info=RUN_CASE,
        case_name=case_name,
        fig_dir=TRAJ_FIG_DIR,
        save_figures=SAVE_FIGURES,
        class_style_map=class_style_map,
        show=True,
    )

if PLOT_POSITION_SNAPSHOTS_COMBINED:
    position_snapshot_figures = phelp.plot_position_snapshots_combined(
        trajectories,
        obs_selections=SNAPSHOT_OBS_SELECTIONS,
        run_title_info=RUN_CASE,
        case_name=case_name,
        fig_dir=TRAJ_FIG_DIR,
        save_figures=SAVE_FIGURES,
        class_style_map=class_style_map,
        show=True,
    )


In [ ]:
# ============================================================
# 7. VORONOÏ CLUSTERING + RESIDENCE TIMES
# ============================================================

voronoi_dir = METRICS_DIR / "voronoi"
residence_dir = METRICS_DIR / "residence"
voronoi_fig_dir = FIG_DIR / "voronoi_snapshots"
pdf_fig_dir = FIG_DIR / "voronoi_pdf"
residence_fig_dir = FIG_DIR / "residence_times"

if SAVE_METRICS:
    voronoi_dir.mkdir(parents=True, exist_ok=True)
    residence_dir.mkdir(parents=True, exist_ok=True)

if SAVE_FIGURES:
    voronoi_fig_dir.mkdir(parents=True, exist_ok=True)
    pdf_fig_dir.mkdir(parents=True, exist_ok=True)
    residence_fig_dir.mkdir(parents=True, exist_ok=True)

reference_ds = next(iter(trajectories.values()))["ds"]
n_obs = reference_ds.sizes["obs"]

def resolve_many(selections):
    out = []
    for selection in selections:
        obs_idx = phelp.resolve_obs_index(selection, n_obs)
        if obs_idx < 0 or obs_idx >= n_obs:
            print(f"Skipping invalid obs selection: {selection}")
            continue
        out.append(obs_idx)
    return sorted(set(out))

snapshot_obs_indices = resolve_many(SNAPSHOT_OBS_SELECTIONS)
pdf_obs_indices = resolve_many(PDF_OBS_SELECTIONS)
residence_obs_indices = list(range(n_obs)) if COMPUTE_RESIDENCE_FOR_ALL_OBS else snapshot_obs_indices

voronoi_outputs = {}
cluster_outputs = {}
residence_outputs = {}

for particle_tag, item in trajectories.items():
    ds = item["ds"]
    label = item.get("display_label", item.get("label", particle_tag))
    style = class_style_map[particle_tag]

    print(f"\nProcessing clustering metrics: {particle_tag} ({label})")

    # Compact selected-snapshot summary.
    summary = mclus.compute_voronoi_summary(
        ds_traj=ds,
        obs_indices=snapshot_obs_indices,
        domain=domain,
    )

    # Monchaux-style Voronoï area PDF.
    pdf = mclus.compute_voronoi_pdf(
        ds_traj=ds,
        obs_indices=pdf_obs_indices,
        domain=domain,
        bins=VORONOI_BINS,
        log_area=VORONOI_LOG_AREA,
    )

    # Particle-level cluster flags for residence-time analysis.
    cluster_ts = mclus.compute_voronoi_clustering_timeseries(
        ds_traj=ds,
        obs_indices=residence_obs_indices,
        domain=domain,
        bins=VORONOI_BINS,
        log_area=VORONOI_LOG_AREA,
        threshold_method=VORONOI_THRESHOLD_METHOD,
        fixed_threshold_area_norm=FIXED_CLUSTER_THRESHOLD_AREA_NORM,
    )

    residence = mclus.compute_cluster_residence_times(
        cluster_ts,
        ds_traj=ds,
        min_duration_obs=MIN_RESIDENCE_DURATION_OBS,
    )
    residence_stats = mclus.residence_summary(residence)

    common_attrs = {
        "case_name": case_name,
        "particle_tag": particle_tag,
        "particle_label": label,
        "level_tag": level_tag,
        "release_time_index": int(RELEASE_TIME_INDEX),
        "runtime_days": float(RUNTIME_DAYS),
    }
    
    summary.attrs.update(common_attrs)
    pdf.attrs.update(common_attrs)
    cluster_ts.attrs.update(common_attrs)
    residence.attrs.update(common_attrs)
    residence_stats.attrs.update(common_attrs)

    voronoi_outputs[particle_tag] = {"summary": summary, "pdf": pdf}
    cluster_outputs[particle_tag] = cluster_ts
    residence_outputs[particle_tag] = {"events": residence, "summary": residence_stats}

    # NetCDF does not accept boolean attributes.
    # Convert only boolean attrs; keep everything else unchanged.
    for out_ds in [summary, pdf, cluster_ts, residence, residence_stats]:
        for key, value in list(out_ds.attrs.items()):
            if isinstance(value, (bool, np.bool_)):
                out_ds.attrs[key] = int(value)

    if SAVE_METRICS:
        summary_path = voronoi_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_summary.nc"
        pdf_path = voronoi_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_pdf.nc"
        cluster_path = voronoi_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_cluster_timeseries.nc"
        residence_path = residence_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_residence_events.nc"
        residence_summary_path = residence_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_residence_summary.nc"

        summary.to_netcdf(summary_path)
        pdf.to_netcdf(pdf_path)
        cluster_ts.to_netcdf(cluster_path)
        residence.to_netcdf(residence_path)
        residence_stats.to_netcdf(residence_summary_path)

        print(f"  saved summary         : {summary_path.name}")
        print(f"  saved PDF             : {pdf_path.name}")
        print(f"  saved cluster flags   : {cluster_path.name}")
        print(f"  saved residence events: {residence_path.name}")

print("\nDone computing clustering/residence metrics.")


In [ ]:
# ============================================================
# 8. VORONOÏ SNAPSHOTS
# ============================================================

if PLOT_VORONOI_SNAPSHOTS:
    for obs_idx in snapshot_obs_indices:
        elapsed_days = phelp.get_elapsed_time_days(reference_ds, obs_idx)

        for particle_tag, item in trajectories.items():
            ds = item["ds"]
            label = item.get("display_label", item.get("label", particle_tag))
            style = class_style_map[particle_tag]
            xname, yname = phelp.get_xy_names(ds)

            x = ds[xname].isel(obs=obs_idx).values
            y = ds[yname].isel(obs=obs_idx).values
            result = mclus.periodic_voronoi_snapshot(x=x, y=y, domain=domain)

            fig, ax = plt.subplots(figsize=phelp.get_figsize("map"), facecolor="white")
            ax.set_facecolor("white")

            mclus.plot_voronoi_snapshot(
                ax=ax,
                pieces=result["pieces"],
                x=x,
                y=y,
                domain=domain,
                line_color=getattr(ptheme, "VORONOI_LINE_COLOR", "0.35"),
                point_color=style["color"],
                point_marker=style["marker"],
                point_size=getattr(ptheme, "VORONOI_POINT_SIZE", 8),
                point_alpha=getattr(ptheme, "VORONOI_POINT_ALPHA", 0.9),
                line_width=getattr(ptheme, "VORONOI_LINE_WIDTH", 0.35),
                line_alpha=getattr(ptheme, "VORONOI_LINE_ALPHA", 0.75),
                km=True,
                label=label,
            )

            ax.set_xlabel("x [km]")
            ax.set_ylabel("y [km]")
            ax.grid(True, alpha=getattr(ptheme, "GRID_ALPHA", 0.35))
            ax.legend(loc="best")
            ax.set_title(phelp.short_time_title(case_name, elapsed_days))
            fig.tight_layout()

            phelp.savefig_if_enabled(
                fig,
                voronoi_fig_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_voronoi_obs{obs_idx:04d}.png",
                save=SAVE_FIGURES,
            )

            plt.show()


In [ ]:
# ============================================================
# 9. VORONOÏ AREA PDF
# ============================================================

if PLOT_VORONOI_PDF:
    for obs_idx in pdf_obs_indices:
        elapsed_days = phelp.get_elapsed_time_days(reference_ds, obs_idx)

        fig, ax = plt.subplots(figsize=phelp.get_figsize("small"), facecolor="white")
        ax.set_facecolor("white")

        first = True
        for particle_tag, item in trajectories.items():
            label = item.get("display_label", item.get("label", particle_tag))
            style = class_style_map[particle_tag]
            pdf = voronoi_outputs[particle_tag]["pdf"]

            mclus.plot_voronoi_pdf(
                pdf,
                obs_idx=obs_idx,
                ax=ax,
                label=label,
                color=style["color"],
                title=None,
                show_poisson=first,
            )
            first = False

        ax.set_title(phelp.short_time_title(case_name, elapsed_days))
        ax.legend(loc="best")
        fig.tight_layout()

        phelp.savefig_if_enabled(
            fig,
            pdf_fig_dir / f"{case_name}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_voronoi_pdf_obs{obs_idx:04d}.png",
            save=SAVE_FIGURES,
        )

        plt.show()


In [ ]:
# ============================================================
# 10. CLUSTER RESIDENCE-TIME PDF
# ============================================================

if PLOT_RESIDENCE_PDF:
    fig, ax = plt.subplots(figsize=phelp.get_figsize("small"), facecolor="white")
    ax.set_facecolor("white")

    for particle_tag, item in trajectories.items():
        label = item.get("display_label", item.get("label", particle_tag))
        style = class_style_map[particle_tag]
        residence = residence_outputs[particle_tag]["events"]

        mclus.plot_residence_time_pdf(
            residence,
            ax=ax,
            bins=30,
            label=label,
            color=style["color"],
            title=None,
        )

    ax.set_title(case_name)
    ax.legend(loc="best")
    fig.tight_layout()

    phelp.savefig_if_enabled(
        fig,
        residence_fig_dir / f"{case_name}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_cluster_residence_pdf.png",
        save=SAVE_FIGURES,
    )

    plt.show()

# Compact summary table
rows = []
for particle_tag, item in trajectories.items():
    label = item.get("display_label", item.get("label", particle_tag))
    rs = residence_outputs[particle_tag]["summary"]
    rows.append({
        "particle_tag": particle_tag,
        "label": label,
        "n_events": int(rs["n_events"].values),
        "mean_days": float(rs["mean_days"].values),
        "median_days": float(rs["median_days"].values),
        "p95_days": float(rs["p95_days"].values),
        "max_days": float(rs["max_days"].values),
    })

import pandas as pd
display(pd.DataFrame(rows))


In [ ]:
# ============================================================
# 11. OPTIONAL FUTURE FLOW-DIAGNOSTIC COUPLING
# ============================================================

# Recommended data layout:
#   z.flow_postprocessing/results/<case_name>/flow/flow_diagnostics.zarr
#       variables: ro(time,y,x), div_f(time,y,x), strain_f(time,y,x), speed(time,y,x)
#       coordinates: time [s], x [m], y [m]
#
#   z.parcels_postprocessing/results/<case_name>/coupled/<particle_tag>_samples.zarr
#       variables: ro_particle, div_f_particle, strain_f_particle, cluster_flag, voronoi_area_norm
#       dimensions: trajectory, obs

COUPLE_TO_FLOW_DIAGNOSTICS = False
FLOW_DIAGNOSTICS_PATH = (
    PROJECT_DIR
    / "z.flow_postprocessing"
    / "results"
    / case_name
    / "flow"
    / "flow_diagnostics.zarr"
)

if COUPLE_TO_FLOW_DIAGNOSTICS:
    if not FLOW_DIAGNOSTICS_PATH.exists():
        raise FileNotFoundError(f"Missing flow diagnostics file:\n{FLOW_DIAGNOSTICS_PATH}")

    ds_flow = mcouple.open_flow_diagnostics(FLOW_DIAGNOSTICS_PATH)
    coupled_dir = CASE_RESULTS_DIR / "coupled"
    coupled_dir.mkdir(parents=True, exist_ok=True)

    for particle_tag, item in trajectories.items():
        ds = item["ds"]
        cluster_ts = cluster_outputs[particle_tag]

        samples = mcouple.sample_flow_at_particles(
            ds_traj=ds,
            ds_flow=ds_flow,
            variables=("ro", "div_f", "strain_f"),
            obs_indices=cluster_ts["obs"].values,
        )
        coupled = mcouple.merge_particle_flow_and_clustering(samples, cluster_ts)

        out_path = coupled_dir / f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}_flow_cluster_samples.zarr"
        coupled.to_zarr(out_path, mode="w")
        print(f"Saved coupled particle-flow samples: {out_path}")
